# **VIS and IR PEDESTRIAN DETECTION BASELINES**

*NOTE:
Experiment status

This notebook implements our preliminary VIS-only and IR-only baselines.
It is inspired by LLVIP's single-modality comparison but does not attempt
to reproduce its exact published settings or scores.

Both detectors use our common development protocol:
YOLOv5s, input size 640, batch size 8, seed 42, and 10 training epochs.

The development-validation set is used for checkpoint selection.
The official test set remains unused.

These experiments establish individual-sensor reference performance.
They do not establish that fusion improves detection.
Our equal-weight fusion baseline is a later experiment.

*Purpose:*

*   Train two YOLOv5s detectors separately: one on visible images and one on
IR images
*   Compare their development-validation performance using matching training settings and paired image membership

*Dataset:*
*   Recovered LLVIP bundle: llvip_milestone_recovered_v1.zip
*   Training: 10,028 pairs
*   Development validation: 1,995 pairs
*   *NOTE: Official Test data is NOT used for training as of now

*Annotation Handling:*
*   Discard 5 zero-width boxes while retaining their image pairs
*   Exclude training pairs 100030 and 100033 from both modalities because
their original annotations contain no objects, and their negative status
was uncertain when inspected visually

*Training Protocol:*
*   Same COCO_pretrained starting checkpoint for both detectors
*   **Configurations**:
    *   Epochs: 10
    *   Image size: 640x640 pixel resolution
    *   Batch size: 8
    *   Seed: 42
*   Same detector revision, hyperparameters, and evaluation proedure

*Checkpoint Selection:*
*   Highest development-validation AP50-95
*   For an exact tie, select the earlier epoch

*Existing Pilot Tests:*
*   A separate two-epoch VIS pilot test completed in 13.96 min on T4 GPU
*   Pilot weights are NOT used to initialize these baseline runs

*Scope:*
*   These are separately trained single-sensor baselines
*   Our equal-weight fusion baseline and adaptive fusion are later experiments



-------------------------------------------------------------------------------

## Mount Drive and Define Paths

In [2]:
from pathlib import Path
from google.colab import drive

# connect this notebook's runtime to google drive
drive.mount("/content/drive")

SHARED_ROOT = Path("/content/drive/MyDrive/EO_IR_Capstone")
assert SHARED_ROOT.is_dir(), (
    "Shared folder not found. Check the Google account and Drive shortcut."
)

# This is the recovered archive we already built and verified
ARCHIVE_DRIVE = (
    SHARED_ROOT
    / "data/prepared/recovery_v1/llvip_milestone_recovered_v1.zip"
)
ARCHIVE_SHA256 = (
    "5c5274dabf94113d66bf86fc81cb86a2b71"
    "a0adfe2e19d5322c28d356b06e07e"
)

assert ARCHIVE_DRIVE.is_file(), f"Archive missing: {ARCHIVE_DRIVE}"

# Keep runtime paths stable so interrupted runs can resume in a fresh session
YOLO_DIR = Path("/content/yolov5")
EXTRACT_DIR = Path("/content/eo_ir_recovery/verified_bundle")
DATA_ROOT = EXTRACT_DIR / "eo_ir_data"

# keep this experiment ID unchanged when resuming these runs
# Use a different ID only when deliberately starting a new experiment
EXPERIMENT_ID = "baselines_recovery_v1_10ep_seed42_v1"
EXPERIMENT_DIR = SHARED_ROOT / "runs" / EXPERIMENT_ID
CONFIG_DIR = EXPERIMENT_DIR / "configs"

print("Shared archive", ARCHIVE_DRIVE)
print("Experiment outputs:", EXPERIMENT_DIR)

Mounted at /content/drive
Shared archive /content/drive/MyDrive/EO_IR_Capstone/data/prepared/recovery_v1/llvip_milestone_recovered_v1.zip
Experiment outputs: /content/drive/MyDrive/EO_IR_Capstone/runs/baselines_recovery_v1_10ep_seed42_v1


## Install and Check Detector

In [7]:
import importlib.metadata
import json
import subprocess
import sys
import torch

# fail before installation or training if this runtime has no GPU
assert torch.cuda.is_available(), (
    "Select a GPU under Runtime --> Change runtime type."
)
print("GPU:", torch.cuda.get_device_name(0))
YOLO_COMMIT = "f9578796b8a64889aa9e66dc06914d19f40d338e"

#Clone public detector code if this is a fresh runtime
if not YOLO_DIR.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/ultralytics/yolov5.git",
            str(YOLO_DIR),
        ],
        check=True,
  )

# use the same detector revision as the successful pilot
subprocess.run(
    ["git", "-C", str(YOLO_DIR), "checkout", "--detach", YOLO_COMMIT],
    check=True,
)

#Preserve installed versions of key packages while satisfying requirements
constraints = Path("/content/eo_ir_runtime_constraints.txt")
protected_packages = [
    "torch", "torchvision", "numpy", "pandas", "pillow", "ultralytics"
]
pins = []

for package in protected_packages:
    try:
        version = importlib.metadata.version(package)
        pins.append(f"{package}=={version}")
    except importlib.metadata.PackageNotFoundError:
        pass

constraints.write_text("\n".join(pins) + "\n")
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "-r", str(YOLO_DIR / "requirements.txt"),
        "-c", str(constraints),
    ],
    check=True,
)

# Verify the checkout and record the actual environment used for these runs
actual_commit = subprocess.check_output(
    ["git", "-C", str(YOLO_DIR), "rev-parse", "HEAD"],
    text=True,
).strip()
assert actual_commit == YOLO_COMMIT

ENVIRONMENT = {
    "python": sys.version.split()[0],
    "gpu": torch.cuda.get_device_name(0),
    "yolov5_commit": actual_commit,
    "packages": {
        package: importlib.metadata.version(package)
        for package in protected_packages
    },
}

print(json.dumps(ENVIRONMENT, indent=2))

GPU: Tesla T4
{
  "python": "3.13.16",
  "gpu": "Tesla T4",
  "yolov5_commit": "f9578796b8a64889aa9e66dc06914d19f40d338e",
  "packages": {
    "torch": "2.11.0+cu130",
    "torchvision": "0.26.0+cu130",
    "numpy": "2.1.3",
    "pandas": "2.2.3",
    "pillow": "11.3.0",
    "ultralytics": "8.4.188"
  }
}


## Load the Recovered Archive

In [8]:
import hashlib
import shutil
import zipfile

def sha256_file(path):
    """Calculate a file's checksum without loading it all into memory."""
    digest = hashlib.sha256()
    with Path(path).open("rb") as file:
        for chunk in iter(lambda: file.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

# Prefer the archive left by our recovery work if this runtime has it.
recovery_copy = Path(
    "/content/eo_ir_recovery/llvip_milestone_recovered_v1.zip"
)
download_copy = Path("/content/llvip_milestone_recovered_v1.zip")

if recovery_copy.is_file():
    local_archive = recovery_copy
else:
    local_archive = download_copy
    if not local_archive.is_file():
        print("Copying recovered archive from Drive...", flush=True)
        shutil.copyfile(ARCHIVE_DRIVE, local_archive)

print("Verifying archive checksum...", flush=True)
assert sha256_file(local_archive) == ARCHIVE_SHA256, (
    "Archive checksum mismatch. Stop before training."
)

# Extract the verified bundle into local storage for fast image loading.
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
print("Extracting prepared data...", flush=True)

with zipfile.ZipFile(local_archive) as archive:
    archive.extractall(EXTRACT_DIR)

assert DATA_ROOT.is_dir()
print("Prepared data available at:", DATA_ROOT)

Copying recovered archive from Drive...
Verifying archive checksum...
Extracting prepared data...
Prepared data available at: /content/eo_ir_recovery/verified_bundle/eo_ir_data


## Validate Data

In [9]:
import json

inventory = json.loads((DATA_ROOT / "inventory.json").read_text())
assert inventory["split"]["train_pairs"] == 10028
assert inventory["split"]["val_pairs"] == 1995

pair_sets = {}

for modality in ["vis", "ir"]:
    for split, expected in [("train", 10028), ("val", 1995)]:
        entries = (DATA_ROOT / f"{modality}_{split}.txt").read_text().split()
        image_paths = [DATA_ROOT / entry for entry in entries]
        ids = [path.stem for path in image_paths]

        # Check membership, duplicates, and the presence of corresponding labels.
        assert len(ids) == expected
        assert len(set(ids)) == expected
        assert all(path.is_file() for path in image_paths)
        assert all(
            (DATA_ROOT / modality / "labels" / f"{p}.txt").is_file()
            for p in ids
        )

        pair_sets[(modality, split)] = set(ids)

assert pair_sets[("vis", "train")] == pair_sets[("ir", "train")]
assert pair_sets[("vis", "val")] == pair_sets[("ir", "val")]
assert not pair_sets[("vis", "train")] & pair_sets[("vis", "val")]
assert not {"100030", "100033"} & (
    pair_sets[("vis", "train")] | pair_sets[("vis", "val")]
)

# Import the loader from the pinned detector checkout.
sys.path.insert(0, str(YOLO_DIR))
from utils.dataloaders import LoadImagesAndLabels

for modality in ["vis", "ir"]:
    for split in ["train", "val"]:
        loaded = LoadImagesAndLabels(
            str(DATA_ROOT / f"{modality}_{split}.txt"),
            img_size=640,
            batch_size=8,
            augment=False,
        )
        assert len(loaded.im_files) == len(pair_sets[(modality, split)])
        print(f"{modality}_{split}: passed")

print("\nDATA VALIDATION PASSED")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Scanning /content/eo_ir_recovery/verified_bundle/eo_ir_data/vis_train... 10028 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 10028/10028 554.0it/s 18.1s
New cache created: /content/eo_ir_recovery/verified_bundle/eo_ir_data/vis_train.cache
vis_train: passed
Scanning /content/eo_ir_recovery/verified_bundle/eo_ir_data/vis_val... 1995 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1995/1995 977.0it/s 2.0s
New cache created: /content/eo_ir_recovery/verified_bundle/eo_ir_data/vis_val.cache
vis_val: passed
Scanning /content/eo_ir_recovery/verified_bundle/eo_ir_data/ir_train... 10028 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 10028/10028 636.0it/s 15.8s
New cache created: /content/eo_ir

## Save the Shared Training Protocol

In [10]:
import urllib.request
import yaml

CONFIG_DIR.mkdir(parents=True, exist_ok=True)

# Obtain the original pretrained checkpoint, never a pilot or trained baseline.
STARTING_WEIGHTS = YOLO_DIR / "yolov5s.pt"

if not STARTING_WEIGHTS.is_file():
    print("Downloading pretrained YOLOv5s weights...", flush=True)
    urllib.request.urlretrieve(
        "https://github.com/ultralytics/yolov5/releases/download/v7.0/"
        "yolov5s.pt",
        STARTING_WEIGHTS,
    )

STARTING_SHA = sha256_file(STARTING_WEIGHTS)

# Both modalities use exactly the same hyperparameter file.
hyp_source = YOLO_DIR / "data/hyps/hyp.scratch-low.yaml"
SHARED_HYP = CONFIG_DIR / "shared_hyp.yaml"

if SHARED_HYP.exists():
    assert SHARED_HYP.read_bytes() == hyp_source.read_bytes(), (
        "Existing experiment hyperparameters differ."
    )
else:
    shutil.copyfile(hyp_source, SHARED_HYP)

DATA_CONFIGS = {}

for modality in ["vis", "ir"]:
    config = {
        "path": str(DATA_ROOT),
        "train": f"{modality}_train.txt",
        "val": f"{modality}_val.txt",
        "nc": 1,
        "names": ["person"],
    }
    path = CONFIG_DIR / f"{modality}.yaml"

    # Refuse to silently change a configuration belonging to an existing run.
    if path.exists():
        assert yaml.safe_load(path.read_text()) == config
    else:
        path.write_text(yaml.safe_dump(config, sort_keys=False))

    DATA_CONFIGS[modality] = path

PROTOCOL = {
    "dataset_sha256": ARCHIVE_SHA256,
    "train_pairs": 10028,
    "val_pairs": 1995,
    "epochs": 10,
    "batch_size": 8,
    "image_size": 640,
    "seed": 42,
    "workers": 2,
    "optimizer": "SGD",
    "starting_weights_sha256": STARTING_SHA,
    "hyperparameters_sha256": sha256_file(SHARED_HYP),
    "checkpoint_selection": "Highest AP50-95; earliest epoch breaks exact ties",
    "environment": ENVIRONMENT,
}

protocol_path = EXPERIMENT_DIR / "protocol.json"

# A resumed experiment must retain its original protocol.
if protocol_path.exists():
    assert json.loads(protocol_path.read_text()) == PROTOCOL, (
        "This experiment's saved protocol differs from the current setup. "
        "Stop and inspect the difference before resuming."
    )
else:
    protocol_path.write_text(json.dumps(PROTOCOL, indent=2))

# Preserve a full package inventory as additional environment documentation.
freeze = subprocess.check_output(
    [sys.executable, "-m", "pip", "freeze"], text=True
)
if not (EXPERIMENT_DIR / "pip_freeze.txt").exists():
    (EXPERIMENT_DIR / "pip_freeze.txt").write_text(freeze)

print("PROTOCOL SAVED:", protocol_path)
print("Both detectors will start from:", STARTING_WEIGHTS)

PROTOCOL SAVED: /content/drive/MyDrive/EO_IR_Capstone/runs/baselines_recovery_v1_10ep_seed42_v1/protocol.json
Both detectors will start from: /content/yolov5/yolov5s.pt


## Define the Shared Training Function (doens't actually start training process)

In [12]:
import os
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd

METRICS = [
    "metrics/precision",
    "metrics/recall",
    "metrics/mAP_0.5",
    "metrics/mAP_0.5:0.95",
]

def read_results(run_dir):
    """
    Read a run's CSV and normalize its column names
    """
    results = pd.read_csv(run_dir / "results.csv")
    results.columns = results.columns.str.strip()
    return results

def summarize_run(modality):
    """
    Select and record the highest-AP50-95 saved epoch
    """
    run_dir = EXPERIMENT_DIR / f"{modality}_baseline"
    results = read_results(run_dir)
    assert len(results) == 10, "Expected all 10 epochs"
    assert list(results["epoch"].astype(int)) == list(range(10))
    assert np.isfinite(results[METRICS].to_numpy(dtype=float)).all()

    #if exact tie, choose earlier epoch
    selected = results.sort_values(
        ["metrics/mAP_0.5:0.95", "epoch"],
        ascending=[False, True],
    ).iloc[0]

    epoch = int(selected["epoch"])
    checkpoint = run_dir / "weights" / f"epoch{epoch}.pt"
    assert checkpoint.is_file(), f"Checkpoint missing: {checkpoint}"

    summary = {
        "modality": modality,
        "epochs_completed": 10,
        "selected_epoch_zero_based": epoch,
        "selected_checkpoint": str(checkpoint),
        **{name: float(selected[name]) for name in METRICS}
    }
    (run_dir / "selection.json").write_text(json.dumps(summary, indent=2))

    print(f"\n{modality.upper()} COMPLETE")
    print(f"Selected epoch: {epoch + 1}/10")
    for name in METRICS:
        print(f"{name}: {100 * summary[name]:.2f}%")
    print("Selected checkpoint:", checkpoint)
    return summary

def train_baseline(modality, resume=False):
    """Start a baseline, or explicitly resume an interrupted baseline."""
    assert modality in ["vis", "ir"]
    assert torch.cuda.is_available()
    assert sha256_file(STARTING_WEIGHTS) == STARTING_SHA

    run_dir = EXPERIMENT_DIR / f"{modality}_baseline"

    # Rerunning the launch cell must not overwrite a completed experiment.
    if (run_dir / "results.csv").is_file():
        existing = read_results(run_dir)
        if (
            len(existing) == 10
            and (run_dir / "weights/epoch9.pt").is_file()
        ):
            return summarize_run(modality)

    if resume:
        # Only load checkpoints produced by our own trusted training runs.
        # A saved epoch includes optimizer state needed for a true resume.
        candidates = list((run_dir / "weights").glob("epoch*.pt"))
        last = run_dir / "weights/last.pt"
        if last.is_file():
            candidates.append(last)

        usable = []
        for path in candidates:
            try:
                checkpoint = torch.load(
                    path, map_location="cpu", weights_only=False
                )
                epoch = int(checkpoint.get("epoch", -1))
                if (
                    0 <= epoch < 9
                    and checkpoint.get("optimizer") is not None
                ):
                    usable.append((epoch, path))
                del checkpoint
            except Exception as error:
                print(f"Could not read {path.name}: {type(error).__name__}")

        assert usable, "No readable resumable checkpoint found."
        saved_epoch, resume_path = max(usable, key=lambda item: item[0])

        # If the CSV was written before an interrupted checkpoint save,
        # preserve a backup and remove rows beyond the recoverable epoch.
        results_path = run_dir / "results.csv"
        if results_path.is_file():
            previous = read_results(run_dir)
            if (previous["epoch"] > saved_epoch).any():
                stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
                shutil.copyfile(
                    results_path, run_dir / f"results_before_resume_{stamp}.csv"
                )
                previous[
                    previous["epoch"] <= saved_epoch
                ].to_csv(results_path, index=False)

        command = [
            sys.executable, "-u", str(YOLO_DIR / "train.py"),
            "--resume", str(resume_path),
        ]
        print(f"Resuming after completed epoch {saved_epoch + 1}.")
    else:
        # An incomplete existing run needs explicit resume, not a restart.
        assert not run_dir.exists(), (
            f"Run already exists: {run_dir}. "
            f"Use train_baseline('{modality}', resume=True) if interrupted."
        )
        run_dir.mkdir(parents=True)

        command = [
            sys.executable, "-u", str(YOLO_DIR / "train.py"),
            "--data", str(DATA_CONFIGS[modality]),
            "--weights", str(STARTING_WEIGHTS),
            "--hyp", str(SHARED_HYP),
            "--img", "640",
            "--batch-size", "8",
            "--epochs", "10",
            "--seed", "42",
            "--workers", "2",
            "--optimizer", "SGD",
            "--device", "0",
            "--patience", "0",
            "--save-period", "1",
            "--project", str(EXPERIMENT_DIR),
            "--name", f"{modality}_baseline",
            "--exist-ok",
        ]

    environment = os.environ.copy()
    environment["WANDB_MODE"] = "disabled"
    environment["COMET_MODE"] = "DISABLED"

    stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    log_path = run_dir / f"console_{stamp}.log"
    (run_dir / f"command_{stamp}.json").write_text(
        json.dumps(command, indent=2)
    )

    start = time.perf_counter()
    print(f"Starting {modality.upper()}. Full log: {log_path}", flush=True)

    # Save detailed output to Drive and print compact progress in Colab.
    with log_path.open("w") as log:
        process = subprocess.Popen(
            command,
            cwd=str(YOLO_DIR),
            env=environment,
            stdout=log,
            stderr=subprocess.STDOUT,
        )
        try:
            while process.poll() is None:
                time.sleep(30)
                completed = 0
                if (run_dir / "results.csv").is_file():
                    try:
                        completed = len(read_results(run_dir))
                    except pd.errors.EmptyDataError:
                        pass
                elapsed = (time.perf_counter() - start) / 60
                print(
                    f"{modality.upper()}: {completed}/10 epochs recorded; "
                    f"{elapsed:.1f} minutes elapsed",
                    flush=True,
                )
        except KeyboardInterrupt:
            # Stop the training subprocess if you interrupt this cell.
            process.terminate()
            process.wait()
            raise

    if process.returncode != 0:
        print(log_path.read_text(errors="replace")[-8000:])
        raise RuntimeError(f"Training failed. Full log: {log_path}")

    return summarize_run(modality)

## TRAIN VIS
Start VIS training from the original COCO-pretrained checkpoint

In [13]:
vis_summary = train_baseline("vis")

Starting VIS. Full log: /content/drive/MyDrive/EO_IR_Capstone/runs/baselines_recovery_v1_10ep_seed42_v1/vis_baseline/console_20261025_011049.log
VIS: 0/10 epochs recorded; 0.5 minutes elapsed
VIS: 0/10 epochs recorded; 1.0 minutes elapsed
VIS: 0/10 epochs recorded; 1.5 minutes elapsed
VIS: 0/10 epochs recorded; 2.0 minutes elapsed
VIS: 0/10 epochs recorded; 2.5 minutes elapsed
VIS: 0/10 epochs recorded; 3.0 minutes elapsed
VIS: 0/10 epochs recorded; 3.5 minutes elapsed
VIS: 0/10 epochs recorded; 4.0 minutes elapsed
VIS: 0/10 epochs recorded; 4.5 minutes elapsed
VIS: 0/10 epochs recorded; 5.0 minutes elapsed
VIS: 0/10 epochs recorded; 5.5 minutes elapsed
VIS: 0/10 epochs recorded; 6.0 minutes elapsed
VIS: 0/10 epochs recorded; 6.5 minutes elapsed
VIS: 1/10 epochs recorded; 7.0 minutes elapsed
VIS: 1/10 epochs recorded; 7.5 minutes elapsed
VIS: 1/10 epochs recorded; 8.0 minutes elapsed
VIS: 1/10 epochs recorded; 8.5 minutes elapsed
VIS: 1/10 epochs recorded; 9.0 minutes elapsed
VIS: 1/10

## TRAIN IR

Start IR independently from the same original pretrained COCO checkpoint
(IR and VIS are still completely separate detectors, but with the same hyperparams)

In [14]:
ir_summary = train_baseline("ir")

Starting IR. Full log: /content/drive/MyDrive/EO_IR_Capstone/runs/baselines_recovery_v1_10ep_seed42_v1/ir_baseline/console_20261025_021522.log
IR: 0/10 epochs recorded; 0.5 minutes elapsed
IR: 0/10 epochs recorded; 1.0 minutes elapsed
IR: 0/10 epochs recorded; 1.5 minutes elapsed
IR: 0/10 epochs recorded; 2.0 minutes elapsed
IR: 0/10 epochs recorded; 2.5 minutes elapsed
IR: 0/10 epochs recorded; 3.0 minutes elapsed
IR: 0/10 epochs recorded; 3.5 minutes elapsed
IR: 0/10 epochs recorded; 4.0 minutes elapsed
IR: 0/10 epochs recorded; 4.5 minutes elapsed
IR: 0/10 epochs recorded; 5.0 minutes elapsed
IR: 0/10 epochs recorded; 5.5 minutes elapsed
IR: 0/10 epochs recorded; 6.0 minutes elapsed
IR: 1/10 epochs recorded; 6.5 minutes elapsed
IR: 1/10 epochs recorded; 7.0 minutes elapsed
IR: 1/10 epochs recorded; 7.5 minutes elapsed
IR: 1/10 epochs recorded; 8.0 minutes elapsed
IR: 1/10 epochs recorded; 8.5 minutes elapsed
IR: 1/10 epochs recorded; 9.0 minutes elapsed
IR: 1/10 epochs recorded; 9.5

## Save Preliminary Comparison Metrics

In [15]:
# Build a comparison from each run's recorded checkpoint selection.
summaries = []

for modality in ["vis", "ir"]:
    selection_file = (
        EXPERIMENT_DIR / f"{modality}_baseline/selection.json"
    )
    assert selection_file.is_file(), (
        f"{modality.upper()} has no completed selection record."
    )
    summaries.append(json.loads(selection_file.read_text()))

comparison = pd.DataFrame(summaries)
comparison.to_csv(
    EXPERIMENT_DIR / "training_validation_summary.csv",
    index=False,
)

# Preserve decimals in the saved CSV; show percentages in the notebook.
display_table = comparison[["modality"] + METRICS].copy()
display_table[METRICS] *= 100

print("SELECTED-EPOCH DEVELOPMENT-VALIDATION METRICS — PERCENT")
print(display_table.to_string(index=False))
print("\nOutputs saved to:", EXPERIMENT_DIR)

SELECTED-EPOCH DEVELOPMENT-VALIDATION METRICS — PERCENT
modality  metrics/precision  metrics/recall  metrics/mAP_0.5  metrics/mAP_0.5:0.95
     vis             91.513          75.432           85.299                44.023
      ir             95.437          89.554           96.025                58.835

Outputs saved to: /content/drive/MyDrive/EO_IR_Capstone/runs/baselines_recovery_v1_10ep_seed42_v1
